<a href="https://colab.research.google.com/github/ttktjmt/mjswan/blob/main/examples/colab/anymal_c_velocity.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🦢 mjswan Tutorial with anymal_c_velocity**

This notebook is an example of the whole workflow of mjswan with mjlab ([anymal_c_velocity](https://github.com/mujocolab/anymal_c_velocity))

<img src="https://github.com/mujocolab/anymal_c_velocity/blob/main/teaser.gif?raw=true" width="300">

## **⚙️ Setup**

In [ ]:
%cd /content
!if [ ! -d "anymal_c_velocity" ]; then git clone -q --recursive https://github.com/mujocolab/anymal_c_velocity.git; fi
%cd /content/anymal_c_velocity
!pip install -e . -q

%cd /content
!if [ ! -d "mjswan" ]; then git clone -q --recursive https://github.com/ttktjmt/mjswan.git; fi
%cd /content/mjswan
!pip install -e . -q

## **✔️ Sanity Check (optional)**

In [ ]:
import re
import subprocess
import sys

process = subprocess.Popen(
    [
        "uv",
        "run",
        "play",
        "Mjlab-Velocity-Flat-Anymal-C",
        "--agent",
        "random",
        "--num_envs",
        "4",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1,
)

detected_port = None

for line in process.stdout:
    print(line, end="")
    sys.stdout.flush()

    # Extract port number from viser output
    port_match = re.search(r":(\d{4})", line)
    if port_match and "viser" in line.lower():
        detected_port = int(port_match.group(1))
        print("\n" + "=" * 34)
        print(f"✅ Server is running on port {detected_port}!")
        print("=" * 34)
        break

In [ ]:
from google.colab import output

port = detected_port if detected_port else 8081
output.serve_kernel_port_as_iframe(port=port, height=600)  # Wait few minutes

## **📚 Train the anymal_c_velocity Policy**

### **🔑 WandB Setup**

In [ ]:
# Online WandB (if you have an account)
!wandb login

# Local WandB
# !wandb offline

### **🏋️ Train the policy**

This will take a couple of hours

In [ ]:
!uv run train Mjlab-Velocity-Flat-Anymal-C --env.scene.num-envs 4096 --agent.max-iterations 2_001

## **♺ Functions to Get Assets**

In [ ]:
import tempfile
from pathlib import Path

import mujoco
import onnx
import torch
from mjlab.scene import Scene
from mjlab.tasks.registry import load_env_cfg

# 1. Scene Spec


def get_mjlab_scene_spec(task_id: str) -> mujoco.MjSpec:
    env_cfg = load_env_cfg(task_id)
    env_cfg.scene.num_envs = 1
    scene = Scene(env_cfg.scene, device="cpu")

    return scene.spec


# 2. ONNX ModelProto


def download_latest_pt(run) -> Path:
    latest_file = None
    latest_step = -1

    for f in run.files():
        if f.name.endswith(".pt"):
            # "model_2000.pt" -> 2000
            try:
                step = int(f.name.replace("model_", "").replace(".pt", ""))
            except ValueError:
                continue
            if step > latest_step:
                latest_step = step
                latest_file = f

    if latest_file is None:
        raise RuntimeError("No .pt file found")

    tmp = tempfile.mkdtemp()
    return Path(latest_file.download(root=tmp, replace=True).name)


def get_onnx_policy(run) -> onnx.ModelProto:
    # Download onnx
    tmp = tempfile.mkdtemp()
    onnx_path = Path(run.file("*.onnx").download(root=tmp, replace=True).name)
    onnx_model = onnx.load(str(onnx_path))

    # Load the latest pt weight
    pt_path = download_latest_pt(run)
    actor_state_dict = torch.load(pt_path, map_location="cpu")["actor_state_dict"]

    # Override weight
    for initializer in onnx_model.graph.initializer:
        if initializer.name in actor_state_dict:
            new_init = onnx.numpy_helper.from_array(
                actor_state_dict[initializer.name].numpy(), name=initializer.name
            )
            onnx_model.graph.initializer.remove(initializer)
            onnx_model.graph.initializer.append(new_init)

    return onnx_model

In [ ]:
import wandb

api = wandb.Api()
run = api.run("ttktjmt-org/mjlab/28eqz4yg")

obj = get_onnx_policy(run)

## **🦢 Generate the mjswan App**

In [ ]:
import wandb

import mjswan

task_id = "Mjlab-Velocity-Flat-Anymal-C"

if wandb.run is not None:
    run = wandb.run
    print(run.name)
else:
    print("Not logged into WandB")
    api = wandb.Api()
    run = api.run("ttktjmt-org/mjlab/28eqz4yg")

APP_DIR = Path("/content/mjswan/app")


builder = mjswan.Builder()
project = builder.add_project(name="anymal_c_velocity")

anymal_c_scene = project.add_scene(spec=get_mjlab_scene_spec(task_id), name=task_id)

anymal_c_scene.add_policy(
    name="anymal_c_velocity",
    policy=get_onnx_policy(run),
).add_velocity_command(
    lin_vel_x=(-2.0, 2.0),
    lin_vel_y=(-2.0, 2.0),
    default_lin_vel_x=0.2,
    default_lin_vel_y=0.2,
)

app = builder.build(output_dir=APP_DIR)
PORT = 8080
app.launch(port=PORT, open_browser=False)

## **🌐 Serve Visualization**

In [ ]:
from google.colab import output

output.serve_kernel_port_as_iframe(PORT, height="700")